# 🧪 ABLATION STUDY — NOTEBOOK 2: Multi-View (360°) Registration

## 🎯 Mục tiêu Thí nghiệm
Ablation test chiến lược **Multi-View 360° Registration**:
- **Biến cố định (Fixed):** Cùng backbone OSNet (`torchreid`, pretrained `Market-1501`) làm verifier — không đổi model.
- **Chiến lược Registration:** Sử dụng dữ liệu quay vòng 360° của người, phân chia thành 4 view groups (`front`, `right`, `back`, `left`) dựa trên Pose Keypoints.
- **Runtime Verification:** Estimate góc nhìn của ứng viên tại runtime → so sánh với embedding của đúng góc nhìn đó (với fallback median khi không xác định được góc).
- **Mục đích:** Đo lường `inter-class gap` (Target vs Strangers) và đối chiếu side-by-side với Notebook 1 (Single-Frame).

---


In [ ]:
# ==============================================================================
# ⚙️ CONFIGURATION — ĐIỀN CÁC THAM SỐ DƯỚI ĐÂY TRƯỚC KHI CHẠY
# ==============================================================================
import os
import sys

# Tự động xác định Project Root (hoạt động tốt khi chạy từ root hoặc trong thư mục notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..")) if "__file__" in locals() else (
    os.path.abspath("..") if os.path.basename(os.path.abspath(".")).lower() == "notebooks" else os.path.abspath(".")
)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 1. Dữ liệu Registration 360° (Hỗ trợ file video .mp4/.avi HOẶC thư mục chứa ảnh .jpg/.png)
MULTIVIEW_DATA_PATH = "PASTE_PATH_HERE" # BẮT BUỘC: Đường dẫn tới video/folder ảnh xoay 360°
NUM_REGISTRATION_FRAMES = 30           # Số lượng frame mẫu cần lấy đều qua 360°

# 2. Test Video (Khuyến nghị dùng LẠI CHÍNH XÁC video từ Notebook 1 để đảm bảo cùng test set)
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
TEST_VIDEO_PATH = os.path.join(DATA_DIR, "raw_video.mp4")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "logs", "notebooks", "run_time", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_CSV_PATH = os.path.join(OUTPUT_DIR, "results_multiview.csv")
NB1_RESULTS_CSV_PATH = os.path.join(OUTPUT_DIR, "results_single_frame.csv") # Đường dẫn file kết quả NB1 để so sánh đối đầu

# 3. Xác nhận Track ID của Target thật sau khi tracking (Xem biểu đồ Section 5 để điền)
KNOWN_TARGET_TRACK_ID = None           # ĐIỀN TRACK ID (ví dụ: 1) sau khi quan sát biểu đồ Section 5

# 4. Cấu hình Models & Thresholds
YOLO_MODEL_PATH = os.path.join(PROJECT_ROOT, "yolo11n.pt")
POSE_MODEL_PATH = os.path.join(PROJECT_ROOT, "yolo11n-pose.pt")   # YOLOv11 Pose / MoveNet
OSNET_VARIANT = "osnet_x1_0"
SIMILARITY_THRESHOLD = 0.70

# 5. Tuỳ chọn xuất video kết quả (Debug Overlay: Green = Target, Red = Untracked / Strangers)
EXPORT_DEBUG_VIDEO = True
DEBUG_VIDEO_OUTPUT = os.path.join(OUTPUT_DIR, "output_multiview_debug.mp4")


In [ ]:
# ==============================================================================
# 📦 CELL 2: DEPENDENCIES, ENVIRONMENT CHECK & INITIALIZATION
# ==============================================================================
import os
import sys
import glob
import cv2
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Đảm bảo import được module trong src/
from src.verifier import OSNetVerifier
from src.view_estimator import ViewEstimator

# 1. Kiểm tra Device (GPU / CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("==================================================")
print(f"🖥️ Execution Device: {device.upper()}")
if device == "cuda":
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
print(f"   PyTorch Version: {torch.__version__}")
print(f"   OpenCV Version: {cv2.__version__}")
print("==================================================")

# 2. Khởi tạo Verifier, Pose Estimator & Detector
print("Loading OSNet Verifier, Pose Estimator & YOLO Detector...")
verifier = OSNetVerifier(variant=OSNET_VARIANT)
detector = YOLO(YOLO_MODEL_PATH)
view_estimator = ViewEstimator(pose_model=POSE_MODEL_PATH)

print("✓ All models initialized successfully!")
print(f"✓ Output Directory: {OUTPUT_DIR}")


---
## Section 1: Load Dữ liệu 360° (Video hoặc Thư mục ảnh)
Tự động nhận diện định dạng dữ liệu (Video .mp4/.avi hay Thư mục ảnh .jpg/.png) và lấy mẫu đều `NUM_REGISTRATION_FRAMES` khung hình.


In [ ]:
# ==============================================================================
# 📂 SECTION 1: LOAD VÀ SAMPLE 360° REGISTRATION FRAMES
# ==============================================================================
if MULTIVIEW_DATA_PATH == "PASTE_PATH_HERE" or not MULTIVIEW_DATA_PATH.strip():
    raise ValueError(
        "❌ LỖI: Bạn chưa điền đường dẫn dữ liệu 360° vào `MULTIVIEW_DATA_PATH` ở Cell 1!\n"
        "Vui lòng cung cấp đường dẫn tới file video .mp4 hoặc thư mục chứa ảnh xoay 360°."
    )

if not os.path.exists(MULTIVIEW_DATA_PATH):
    raise FileNotFoundError(f"Đường dẫn không tồn tại: '{MULTIVIEW_DATA_PATH}'")

sampled_frames = []

if os.path.isdir(MULTIVIEW_DATA_PATH):
    print(f"📁 Phát hiện thư mục ảnh tại: '{MULTIVIEW_DATA_PATH}'")
    img_files = sorted(glob.glob(os.path.join(MULTIVIEW_DATA_PATH, "*.jpg")) +
                       glob.glob(os.path.join(MULTIVIEW_DATA_PATH, "*.png")))
    if not img_files:
        raise ValueError(f"Không tìm thấy ảnh .jpg/.png nào trong thư mục '{MULTIVIEW_DATA_PATH}'!")

    # Lấy mẫu đều NUM_REGISTRATION_FRAMES ảnh
    indices = np.linspace(0, len(img_files) - 1, min(NUM_REGISTRATION_FRAMES, len(img_files)), dtype=int)
    for idx in indices:
        img = cv2.imread(img_files[idx])
        if img is not None:
            sampled_frames.append(img)
    print(f"✓ Đã load {len(sampled_frames)} ảnh từ tổng số {len(img_files)} ảnh.")

else:
    print(f"🎥 Phát hiện file video tại: '{MULTIVIEW_DATA_PATH}'")
    cap = cv2.VideoCapture(MULTIVIEW_DATA_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"Không thể mở video tại '{MULTIVIEW_DATA_PATH}'!")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(0, total_frames - 1), min(NUM_REGISTRATION_FRAMES, total_frames), dtype=int)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret and frame is not None:
            sampled_frames.append(frame)
    cap.release()
    print(f"✓ Đã lấy mẫu {len(sampled_frames)} frames từ tổng số {total_frames} frames của video.")

if len(sampled_frames) < 4:
    raise ValueError(f"Số lượng frame lấy được quá ít ({len(sampled_frames)} < 4) để thực hiện multi-view registration!")


---
## Section 2: Phân chia View Groups (Front / Right / Back / Left)
Chạy YOLO detect người trên từng frame đăng ký (bắt buộc đúng 1 người), trích xuất pose keypoints, tính góc xoay và phân nhóm vào 4 views:
- `front`: [315°, 360°) ∪ [0°, 45°)
- `right`: [45°, 135°)
- `back`: [135°, 225°)
- `left`: [225°, 315°)


In [ ]:
# ==============================================================================
# 📐 SECTION 2: POSE-BASED 4-VIEW CLASSIFICATION
# ==============================================================================
view_crops = {'front': [], 'right': [], 'back': [], 'left': []}
view_angles = {'front': [], 'right': [], 'back': [], 'left': []}
skipped_count = 0

print("Đang xử lý phân loại góc nhìn cho các frame đăng ký...")

for i, frame in enumerate(sampled_frames):
    # Detect người (class 0)
    results = detector(frame, classes=[0], verbose=False)
    if not results or len(results) == 0 or results[0].boxes is None:
        skipped_count += 1
        continue

    boxes = results[0].boxes
    if len(boxes) != 1:
        # Điều kiện registration: đúng 1 người trong khung hình
        skipped_count += 1
        continue

    x1, y1, x2, y2 = boxes[0].xyxy[0].cpu().numpy().astype(int)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
    crop = frame[y1:y2, x1:x2]

    if crop.size == 0 or crop.shape[0] < 15 or crop.shape[1] < 15:
        skipped_count += 1
        continue

    # Estimate view & angle qua ViewEstimator
    view_name, angle = view_estimator.estimate_view_from_crop(crop)

    if view_name in view_crops:
        view_crops[view_name].append(crop)
        if angle is not None:
            view_angles[view_name].append(angle)
    else:
        # Fallback default nếu pose không tự tin
        view_crops['front'].append(crop)

print("==================================================")
print(f"📊 KẾT QUẢ PHÂN CHIA VIEW GROUPS:")
print(f"  • Tổng frames lấy mẫu: {len(sampled_frames)}")
print(f"  • Số frame bị bỏ qua (0 hoặc >1 người/lỗi): {skipped_count}")
print(f"  • Số frame hợp lệ: {len(sampled_frames) - skipped_count}")
print("--------------------------------------------------")
for v in ['front', 'right', 'back', 'left']:
    count = len(view_crops[v])
    avg_ang = f"{np.mean(view_angles[v]):.1f}°" if view_angles[v] else "N/A"
    status = "✓ OK" if count > 0 else "⚠️ WARNING: RỖNG (0 frames)"
    print(f"  • View '{v.upper():<5}': {count:2d} frames (Góc trung bình: {avg_ang:<6}) -> {status}")
print("==================================================")


---
## Section 3: Build Multi-View Reference Embeddings
Tính toán vector embedding trung bình L2-normalized cho từng View Group.


In [ ]:
# ==============================================================================
# 🧬 SECTION 3: BUILD MULTI-VIEW REFERENCE EMBEDDINGS
# ==============================================================================
reference_embeddings_multiview = {}
all_valid_embeddings = []

for view_name in ['front', 'right', 'back', 'left']:
    crops = view_crops[view_name]
    if len(crops) > 0:
        emb_list = []
        for c in crops:
            emb = verifier.extract(c)
            norm = np.linalg.norm(emb)
            if norm > 1e-6:
                emb = emb / norm
                emb_list.append(emb)
                all_valid_embeddings.append(emb)

        if emb_list:
            mean_vec = np.mean(emb_list, axis=0)
            norm_mean = np.linalg.norm(mean_vec)
            reference_embeddings_multiview[view_name] = (mean_vec / norm_mean) if norm_mean > 1e-6 else mean_vec
        else:
            reference_embeddings_multiview[view_name] = None
    else:
        reference_embeddings_multiview[view_name] = None

if not all_valid_embeddings:
    raise RuntimeError("❌ Không trích xuất được embedding hợp lệ nào từ toàn bộ dữ liệu đăng ký!")

# Composite Mean làm fallback cho các view bị thiếu
composite_mean = np.mean(all_valid_embeddings, axis=0)
composite_mean = composite_mean / np.linalg.norm(composite_mean)

# Gán composite mean cho các view rỗng để đảm bảo hệ thống không bị crash
for v in ['front', 'right', 'back', 'left']:
    if reference_embeddings_multiview[v] is None:
        reference_embeddings_multiview[v] = composite_mean

print("==================================================")
print("✓ MULTI-VIEW REFERENCE EMBEDDINGS HOÀN TẤT:")
for v, emb in reference_embeddings_multiview.items():
    print(f"  • View {v.upper():<5}: shape={emb.shape}, norm={np.linalg.norm(emb):.4f}")
print("==================================================")


---
## Section 4: Track & Multi-View Verify qua Test Video
Chạy YOLOv11 + ByteTrack trên cùng Test Video. Tại mỗi frame, trích xuất crop của ứng viên, estimate view và so sánh Cosine Similarity với embedding đúng view.


In [ ]:
# ==============================================================================
# 🚀 SECTION 4: RUN BYTETRACK & MULTI-VIEW VERIFICATION
# ==============================================================================
if not os.path.exists(TEST_VIDEO_PATH):
    raise FileNotFoundError(f"Không tìm thấy Test Video tại: '{TEST_VIDEO_PATH}'. Hãy đảm bảo đã chạy Section 1 của Notebook 1!")

cap = cv2.VideoCapture(TEST_VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

records = []
frame_idx = 0
start_time = time.time()

print(f"Bắt đầu Multi-View tracking và verification trên {total_frames} frames của '{TEST_VIDEO_PATH}'...")

# Tuỳ chọn ghi video debug (Green bbox = Target, Red bbox = Untracked / Strangers)
video_writer = None
if EXPORT_DEBUG_VIDEO:
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(DEBUG_VIDEO_OUTPUT, fourcc, fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    track_results = detector.track(frame, persist=True, classes=[0], verbose=False)

    if track_results and len(track_results) > 0 and track_results[0].boxes is not None:
        boxes = track_results[0].boxes
        for box in boxes:
            if box.id is None:
                continue

            tid = int(box.id[0].cpu().numpy())
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

            x1_c, y1_c = max(0, x1), max(0, y1)
            x2_c, y2_c = min(frame.shape[1], x2), min(frame.shape[0], y2)
            crop = frame[y1_c:y2_c, x1_c:x2_c]

            if crop.size > 0 and crop.shape[0] > 10 and crop.shape[1] > 10:
                # 1. Trích xuất embedding ứng viên
                emb = verifier.extract(crop)

                # 2. Estimate góc nhìn của ứng viên
                est_view, angle = view_estimator.estimate_view_from_crop(crop)

                # 3. So sánh với đúng reference embedding của view đó (hoặc fallback)
                ref_emb = reference_embeddings_multiview.get(est_view)
                if ref_emb is not None:
                    sim_score = verifier.compare(emb, ref_emb)
                else:
                    # Fallback median qua các view có sẵn
                    available_embs = [e for e in reference_embeddings_multiview.values() if e is not None]
                    sims = [verifier.compare(emb, e) for e in available_embs]
                    sim_score = float(np.median(sims))

                if KNOWN_TARGET_TRACK_ID is not None:
                    is_target = (tid == KNOWN_TARGET_TRACK_ID)
                else:
                    is_target = (sim_score >= SIMILARITY_THRESHOLD)

                records.append({
                    "frame_idx": frame_idx,
                    "track_id": tid,
                    "similarity_score": round(float(sim_score), 4),
                    "estimated_view": est_view or "unknown",
                    "is_target": bool(is_target),
                    "bbox_x1": x1,
                    "bbox_y1": y1,
                    "bbox_x2": x2,
                    "bbox_y2": y2
                })

                if EXPORT_DEBUG_VIDEO:
                    # 🟢 Green cho Target, 🔴 Red cho Untracked / Strangers
                    if is_target:
                        color = (0, 255, 0)
                        thickness = 3
                        label = f"TARGET ID:{tid} V:{est_view or '?'} S:{sim_score:.2f}"
                    else:
                        color = (0, 0, 255)
                        thickness = 2
                        label = f"ID:{tid} V:{est_view or '?'} S:{sim_score:.2f}"

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
                    (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(frame, (x1, max(0, y1 - 20)), (x1 + lw, y1), color, -1)
                    cv2.putText(frame, label, (x1, max(14, y1 - 4)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

    # Header Overlay
    if EXPORT_DEBUG_VIDEO and video_writer:
        cv2.putText(frame, f"Frame: {frame_idx}/{total_frames} | Multi-View (360) Re-ID", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
        video_writer.write(frame)

    frame_idx += 1
    if frame_idx % 50 == 0 or frame_idx == total_frames:
        elapsed = time.time() - start_time
        print(f"  • Đã xử lý {frame_idx}/{total_frames} frames ({frame_idx/total_frames*100:.1f}%) — {frame_idx/elapsed:.1f} FPS")

cap.release()
if video_writer:
    video_writer.release()
    print(f"✓ Video debug đã lưu tại: {DEBUG_VIDEO_OUTPUT}")

# Lưu kết quả CSV
df_mv = pd.DataFrame(records)
df_mv.to_csv(RESULTS_CSV_PATH, index=False)
print(f"✓ Đã lưu toàn bộ {len(df_mv)} detections vào file CSV: '{RESULTS_CSV_PATH}'")


---
## Section 5: Visualize, Phân tích & Bảng So Sánh Đối Đầu (Head-to-Head)
1. Vẽ biểu đồ biến thiên similarity của Multi-View (cùng format trục đồ thị như NB1).
2. So sánh trực tiếp các chỉ số thống kê giữa **Single-Frame (NB1)** và **Multi-View (NB2)** để đánh giá giá trị thực tế của Multi-View Registration.


In [ ]:
# ==============================================================================
# 📊 SECTION 5: VISUALIZE & HEAD-TO-HEAD COMPARISON TABLE
# ==============================================================================
if not os.path.exists(RESULTS_CSV_PATH):
    raise FileNotFoundError(f"Không tìm thấy file CSV: '{RESULTS_CSV_PATH}'")

df_mv = pd.read_csv(RESULTS_CSV_PATH)
unique_tracks = df_mv["track_id"].unique()
print(f"Tổng số track_id trong Multi-View test: {len(unique_tracks)} (Tracks: {list(unique_tracks)})")

# 1. BIỂU ĐỒ LINE CHART: MULTI-VIEW SIMILARITY THEO THỜI GIAN
plt.figure(figsize=(15, 6))
for tid in sorted(unique_tracks):
    track_df = df_mv[df_mv["track_id"] == tid]
    if len(track_df) >= 3:
        plt.plot(track_df["frame_idx"], track_df["similarity_score"], label=f"Track ID {tid}", marker='.', markersize=4, alpha=0.8)

plt.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Ref Threshold (0.80)')
plt.title("Multi-View OSNet Similarity Score qua từng Frame (Mỗi đường là 1 Track ID)", fontsize=14)
plt.xlabel("Frame Index", fontsize=12)
plt.ylabel("Cosine Similarity Score [-1.0, 1.0]", fontsize=12)
plt.ylim(-0.2, 1.05)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper right", bbox_to_anchor=(1.15, 1.0), fontsize=9)
plt.tight_layout()
plt.show()

# 2. PHÂN BỐ VIEW CỦA TỪNG TRACK
print("\nPhân bố các góc nhìn (estimated_view) được nhận diện:")
print(df_mv["estimated_view"].value_counts().to_string())

# 3. BẢNG SO SÁNH ĐỐI ĐẦU: SINGLE-FRAME (NB1) VS MULTI-VIEW (NB2)
print("\n" + "="*75)
print("⚔️ BẢNG SO SÁNH ĐỐI ĐẦU: SINGLE-FRAME (NB1) VS MULTI-VIEW (NB2)")
print("="*75)

if KNOWN_TARGET_TRACK_ID is not None:
    # Multi-View metrics
    t_mv = df_mv[df_mv["track_id"] == KNOWN_TARGET_TRACK_ID]["similarity_score"]
    s_mv = df_mv[df_mv["track_id"] != KNOWN_TARGET_TRACK_ID]["similarity_score"]

    t_mean_mv, t_std_mv = t_mv.mean(), t_mv.std()
    s_mean_mv, s_std_mv = s_mv.mean(), s_mv.std()
    gap_mv = t_mean_mv - s_mean_mv

    # Đọc kết quả từ NB1 CSV nếu có
    if os.path.exists(NB1_RESULTS_CSV_PATH):
        df_sf = pd.read_csv(NB1_RESULTS_CSV_PATH)
        t_sf = df_sf[df_sf["track_id"] == KNOWN_TARGET_TRACK_ID]["similarity_score"]
        s_sf = df_sf[df_sf["track_id"] != KNOWN_TARGET_TRACK_ID]["similarity_score"]

        t_mean_sf, t_std_sf = t_sf.mean(), t_sf.std()
        s_mean_sf, s_std_sf = s_sf.mean(), s_sf.std()
        gap_sf = t_mean_sf - s_mean_sf

        delta_gap = gap_mv - gap_sf

        comp_data = {
            "Metric": [
                "Target Mean Similarity",
                "Target Std Dev (Độ phân tán)",
                "Stranger Mean Similarity",
                "Stranger Std Dev",
                "⭐ Inter-Class Separation Gap (Target - Stranger)",
                "Target Frame Count (N)",
                "Stranger Frame Count (N)"
            ],
            "Single-Frame (NB1)": [
                f"{t_mean_sf:.4f}",
                f"{t_std_sf:.4f}",
                f"{s_mean_sf:.4f}",
                f"{s_std_sf:.4f}",
                f"{gap_sf:+.4f}",
                len(t_sf),
                len(s_sf)
            ],
            "Multi-View (NB2)": [
                f"{t_mean_mv:.4f}",
                f"{t_std_mv:.4f}",
                f"{s_mean_mv:.4f}",
                f"{s_std_mv:.4f}",
                f"{gap_mv:+.4f}",
                len(t_mv),
                len(s_mv)
            ],
            "Delta (NB2 - NB1)": [
                f"{t_mean_mv - t_mean_sf:+.4f}",
                f"{t_std_mv - t_std_sf:+.4f}",
                f"{s_mean_mv - s_mean_sf:+.4f}",
                f"{s_std_mv - s_std_sf:+.4f}",
                f"{delta_gap:+.4f}",
                f"{len(t_mv) - len(t_sf):+d}",
                f"{len(s_mv) - len(s_sf):+d}"
            ]
        }
        df_comp = pd.DataFrame(comp_data)
        print(df_comp.to_string(index=False))
        print("-" * 75)
        print(f"📌 ĐÁNH GIÁ KẾT LUẬN:")
        if delta_gap > 0.05:
            print(f"✓ Multi-View cải thiện đáng kể độ tách biệt lớp (Inter-class gap tăng {delta_gap:+.4f}).")
            print("  -> Chiến lược Multi-View Registration mang lại giá trị thực sự rõ rệt.")
        elif delta_gap > 0:
            print(f"~ Multi-View có cải thiện nhẹ (Inter-class gap tăng {delta_gap:+.4f}), nhưng mức độ khiêm tốn.")
        else:
            print(f"⚠️ Multi-View KHÔNG cải thiện so với Single-Frame (Inter-class gap delta: {delta_gap:+.4f}).")
            print("  -> Single-Frame registration (rẻ hơn, không cần xoay 360°) là lựa chọn đủ dùng.")
    else:
        print(f"⚠️ Không tìm thấy file '{NB1_RESULTS_CSV_PATH}'. Hiển thị số liệu Multi-View độc lập:")
        print(f"  • Target Mean: {t_mean_mv:.4f} (std={t_std_mv:.4f})")
        print(f"  • Stranger Mean: {s_mean_mv:.4f} (std={s_std_mv:.4f})")
        print(f"  • Inter-Class Gap: {gap_mv:+.4f}")
else:
    print("👉 Hãy điền `KNOWN_TARGET_TRACK_ID = <số>` ở Cell 1 và chạy lại để tạo Bảng So Sánh Đối Đầu.")
print("="*75)


---
## Section 6: Xuất Video Kết Quả Hoàn Chỉnh (Annotated Multi-View Video)
Tạo video chất lượng cao với HUD Status Bar:
- 🟢 **Bounding box Xanh lá (Green)**: Cho Target Person (được chỉ định hoặc nhận diện chính xác nhất).
- 🔴 **Bounding box Đỏ (Red)**: Cho người không được chọn (Untracked / Strangers).
- Hiển thị góc nhìn phân loại (`front`, `right`, `back`, `left`) và Cosine Similarity của từng cá nhân theo thời gian thực.



In [ ]:
# ==============================================================================
# 🎬 SECTION 6: RENDER ANNOTATED MULTI-VIEW VIDEO
# ==============================================================================
ANNOTATED_MV_VIDEO_PATH = os.path.join(OUTPUT_DIR, "output_multiview_annotated.mp4")

if not os.path.exists(RESULTS_CSV_PATH):
    raise FileNotFoundError(f"Không tìm thấy file kết quả CSV: '{RESULTS_CSV_PATH}'. Hãy chạy Section 4 trước!")

df_res = pd.read_csv(RESULTS_CSV_PATH)
if df_res.empty:
    raise ValueError("Dữ liệu kết quả Multi-View trống!")

active_target_id = KNOWN_TARGET_TRACK_ID
if active_target_id is None:
    avg_scores = df_res.groupby("track_id")["similarity_score"].agg(["mean", "count"])
    avg_scores = avg_scores[avg_scores["count"] >= 5]
    if not avg_scores.empty:
        active_target_id = int(avg_scores["mean"].idxmax())
        print(f"ℹ️ Tự động nhận diện Multi-View Target Track ID: ID = {active_target_id} (Mean Sim: {avg_scores.loc[active_target_id, 'mean']:.4f})")
    else:
        active_target_id = int(df_res.groupby("track_id")["similarity_score"].mean().idxmax())
        print(f"ℹ️ Chọn Track ID cao nhất: ID = {active_target_id}")
else:
    print(f"ℹ️ Sử dụng KNOWN_TARGET_TRACK_ID đã chỉ định: ID = {active_target_id}")

cap = cv2.VideoCapture(TEST_VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(ANNOTATED_MV_VIDEO_PATH, fourcc, fps, (w, h))

print(f"Đang render video chú thích Multi-View tới '{ANNOTATED_MV_VIDEO_PATH}'...")

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    frame_df = df_res[df_res["frame_idx"] == frame_idx]

    target_present = False
    for _, row in frame_df.iterrows():
        tid = int(row["track_id"])
        sim = float(row["similarity_score"])
        v_name = str(row.get("estimated_view", "?"))
        x1, y1 = int(row["bbox_x1"]), int(row["bbox_y1"])
        x2, y2 = int(row["bbox_x2"]), int(row["bbox_y2"])

        is_target = (tid == active_target_id)
        if is_target:
            target_present = True
            # 🟢 GREEN CHO TARGET
            box_color = (0, 255, 0)
            thickness = 3
            label = f"TARGET ID:{tid} [{v_name}] S:{sim:.2f}"
        else:
            # 🔴 RED CHO UNTRACKED / STRANGER
            box_color = (0, 0, 255)
            thickness = 2
            label = f"ID:{tid} [{v_name}] S:{sim:.2f}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, thickness)
        (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(frame, (x1, max(0, y1 - 22)), (x1 + lw, y1), box_color, -1)
        cv2.putText(frame, label, (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

    # HUD Banner
    hud_bg = frame.copy()
    cv2.rectangle(hud_bg, (0, 0), (w, 50), (20, 20, 20), -1)
    cv2.addWeighted(hud_bg, 0.6, frame, 0.4, 0, frame)

    status_text = f"MULTI-VIEW TARGET ID: {active_target_id}" if target_present else "TARGET: SEARCHING / OCCLUDED"
    status_color = (0, 255, 0) if target_present else (0, 165, 255)
    cv2.putText(frame, status_text, (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2, cv2.LINE_AA)
    cv2.putText(frame, f"Frame: {frame_idx}/{total_frames}", (w - 240, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    out_writer.write(frame)
    frame_idx += 1

cap.release()
out_writer.release()

print("="*60)
print(f"✓ Video Multi-View chú thích đã được lưu thành công tại:")
print(f"  👉 {ANNOTATED_MV_VIDEO_PATH}")
print(f"  • 🟢 Xanh lá: Multi-View Target Person (ID {active_target_id})")
print(f"  • 🔴 Đỏ: Untracked / Strangers")
print("="*60)

